# Agência Nacional de Energia Elétrica (ANEEL)

O site da ANEEL apresenta uma sessão de [**Informações Geográficas**](https://www.aneel.gov.br/informacoes-geograficas). Analisando o material, costata-se que eles utilizam a estrutura do _ArcGIS Server_, em um portal denominado _Sistema de Informações Georreferenciadas do Setor Elétrico (SIGEL)_.

- https://sigel.aneel.gov.br/arcgis/rest/

<br>

---

## Siglas

<br>

A instituição tem várias siglas que estão apresentadas nos dados do ArcGIS.

**Superintendências**

- SRM: Superintendência de Regulação Econômica e Estudos de Mercado
- SMA: Superintendência de Mediação Administrativa, Ouvidoria Setorial e Participação Pública
- SFE: Superintendência de Fiscalização dos Serviços de Eletricidade
- SFG: Superintendência de Fiscalização dos Serviços de Geração

<br>

**Outras**

- SIPH: Sistema de Informações do Potencial Hidroelétrico
- GGT: Gestão Geoespacializada da Transmissão
- SGO: Sistema de Gestão de Ouvidoria
- IASC: Índice ANEEL de Satisfação do Consumidor
- UFV: Centrais Geradoras Fotovoltaicas
- EOL: Usinas Eólicas
- UTN: Usina Eletronuclear
- UTE: Usinas Termelétricas
- PCH: Pequenas Centrais Hidrelétricas
- AHE: Aproveitamentos Hidrelétricos
- UHE: Usinas Hidrelétricas
- SKATE

<br>

---

## _Download_ de Dados

Por meio do acesso ao site de _download_ dos dados, é possível observar a interface e _layers_ disponíveis. Observou-se que são os mesmos _layers_ disponíveis na pasta "Portal" do _webservice_.

![ANEEL](https://i.imgur.com/JoQs2ZT.png)


Para os pacotes que usam python, é necessário

In [ ]:
#!pip3 install arcgis

Definir a variável de ambiente `RESTAPI_USE_ARCPY` como `FALSE` é necessário para evitar que a biblioteca `restapi` mande mensagens de erro ou tente usar o `ArcPy`, que só está disponível para quem tem licença da ESRI.


In [ ]:
import os

os.environ['RESTAPI_USE_ARCPY'] = 'FALSE'

In [ ]:
import warnings

import requests
import restapi
from restapi import NAME, SERVICES, TYPE, ArcServer

import open_geodata as geo

In [ ]:
import json
import pprint
import tempfile
from pathlib import Path

import geopandas as gpd

# from arcgis.raster.functions import *

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
session = requests.Session()
client = restapi.RequestClient(session)
restapi.set_request_client(client)

In [ ]:
# # connect to esri's sample server 6
# url = 'https://sigel.aneel.gov.br/arcgis/rest/services'
# url

In [ ]:
# Connect to restapi.ArcServer instance
ags = restapi.ArcServer(url='https://sigel.aneel.gov.br/arcgis/rest/services')
ags

<br>

Com o uso do rest

In [ ]:
for x in ags.list_services():
    print(x, type(x))

In [ ]:
for root, services in ags.walk(ignore_folder_auth=True):
    print(f'Pasta: {root}')
    # print('\n'.join(f'- {item}' for item in services))
    for service in services:
        print(f'- {service}')

    print(f'-' * 60)

<br>

Obtem detalhes do serviço

In [ ]:
# Listas de Tipos de Serviço
ags.featureServices

In [ ]:
service = ags.getService(name_or_wildcard='SP')
service.name

In [ ]:
# Shapefile
print(service.name)
print(service.description)

# URL
print(service.url)

# Representação
print(repr(service))

# Formatos
print(service.supportedQueryFormats)

# Path
print(service.servicePath)

# documentInfo
print(service.documentInfo)

# Descrição
print(service.description)

# Informações do datum
print(service.initialExtent)
print(service.spatialReference)

In [ ]:
lyr = service.get_layer_url(name='ZEE')
type(lyr)

In [ ]:
# Seleciona Layer no Serviço
lyr = service.layer(name_or_id=19)
type(lyr)

In [ ]:
lyr_query = lyr.query(
    where='1=1',
    # Se exceed_limit=True, retorna todos os registros
    # Se exceed_limit=False, retorna apenas os primeiros 1000 registros
    exceed_limit=True,
    # ------------------------------------
    # Número de registros a serem retornados
    # Se records=None, retorna todos os registros
    # records=10,
    # ------------------------------------
    # Option to return a generator with a FeatureSet in chunks of each query group.
    # Use this to avoid memory errors when fetching many features. Defaults to False
    fetch_in_chunks=True,
)

lyr_query.geometryType

In [ ]:
# shp_file
temp_dir = tempfile.TemporaryDirectory()

In [ ]:
# Cria o caminho temporário em formato Path
temp_path = Path(temp_dir.name)
temp_path

In [ ]:
# Crio pasta temporária
temp_path = Path(tempfile.gettempdir()) / 'open_geodata' / 'aneel'
temp_path.mkdir(exist_ok=True)
temp_path

In [ ]:
# ddd
restapi.exportFeatureSet(
    feature_set=lyr_query,
    #
    out_fc=str(temp_path / 'temp.shp'),
)

In [ ]:
# Read Data
gdf = gpd.read_file(filename=temp_path / 'temp.shp')

# Results
gdf.info()
gdf.head()

<br>

-----

## ArcGIS


In [ ]:
from arcgis import geometry
from arcgis.geocoding import geocode
from arcgis.gis import GIS

In [ ]:
#url = "https://mapas.agenciapcj.org.br/arcgis/rest/services"
 
url = service.url
# Connect to the portal
gis = GIS(url)

In [ ]:
# Search for all feature services and feature collections in the portal
items = gis.content.search(query='type: "Feature Service" OR type: "Feature Collection"', max_items=5000)
items